In [1]:
import json
import re
import pathlib
import multiprocessing as mp
import xarray as xr
import tqdm

import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from functools import partial
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

In [2]:
src_path = pathlib.Path('/gpfs/work2/0/prjs1027/swn-runs/02-2022/computations')
boundary_path = pathlib.Path('/gpfs/work2/0/prjs1027/guus/data/BoundaryConditions/bnd.nc')
write_dir = pathlib.Path('/gpfs/work2/0/prjs1027/tfrecords/run2')
write_dir.mkdir(exist_ok=True, parents=True)
filtered_paths_50 = [path for path in src_path.glob('ens*') if re.search(r'\d+$', str(path))]
# filtered_paths = [path for path in src_path.glob('ens*') if re.search(r'\\b([0-9]|1[0-9]|2[0-4])\\b', str(path))]

In [3]:
filtered_paths = []
for path in filtered_paths_50:
    ens = re.search(r'\d+$', str(path)).group(0)
    if int(ens) in range(25):
        filtered_paths.append(path)
filtered_paths

[PosixPath('/gpfs/work2/0/prjs1027/swn-runs/02-2022/computations/ens16'),
 PosixPath('/gpfs/work2/0/prjs1027/swn-runs/02-2022/computations/ens19'),
 PosixPath('/gpfs/work2/0/prjs1027/swn-runs/02-2022/computations/ens17'),
 PosixPath('/gpfs/work2/0/prjs1027/swn-runs/02-2022/computations/ens8'),
 PosixPath('/gpfs/work2/0/prjs1027/swn-runs/02-2022/computations/ens23'),
 PosixPath('/gpfs/work2/0/prjs1027/swn-runs/02-2022/computations/ens9'),
 PosixPath('/gpfs/work2/0/prjs1027/swn-runs/02-2022/computations/ens18'),
 PosixPath('/gpfs/work2/0/prjs1027/swn-runs/02-2022/computations/ens10'),
 PosixPath('/gpfs/work2/0/prjs1027/swn-runs/02-2022/computations/ens22'),
 PosixPath('/gpfs/work2/0/prjs1027/swn-runs/02-2022/computations/ens4'),
 PosixPath('/gpfs/work2/0/prjs1027/swn-runs/02-2022/computations/ens5'),
 PosixPath('/gpfs/work2/0/prjs1027/swn-runs/02-2022/computations/ens21'),
 PosixPath('/gpfs/work2/0/prjs1027/swn-runs/02-2022/computations/ens6'),
 PosixPath('/gpfs/work2/0/prjs1027/swn-runs

In [4]:
boundary_ds = xr.open_dataset(boundary_path)
dummy_ds = xr.open_dataset(next(filtered_paths[0].glob('*/output/*.nc')))

In [5]:
output_variables = ["hs", "theta0_x", "theta0_y", "tmm10", "tps", "hswe"]
input_variables = ["xwnd", "ywnd", "ssh", "xcur", "ycur"]
boundary_variables = ["hs", "hswe", "tmm10"]
conversion_dict = {"hs": "Hm0", "hswe": "HE10", "tmm10": "Tmm10"}

In [6]:
# %%
def compute_min_max(path):
    all_datasets = []
    min_values = {}
    max_values = {}
    for var in dummy_ds.variables:
        min_values[var] = []
        max_values[var] = []
    
    for _ in tqdm.tqdm(list(path.glob('*'))):
        try:
            ds_path = next(_.glob('output/*.nc'))
            ds = xr.open_dataset(ds_path)

            for var in ds.variables:
                min_values[var].append(ds[var].min().values)
                max_values[var].append(ds[var].max().values)
        except:
            continue

    min_max_values = {}
    for var in dummy_ds.variables:
        min_value = np.min(min_values[var])
        max_value = np.max(max_values[var])
        min_max_values[var] = np.array([min_value, max_value])
    return min_max_values


In [7]:
min_max_values = {}
for paths in filtered_paths:
    min_max_values[paths.stem] = compute_min_max(paths)

100%|██████████| 364/364 [03:20<00:00,  1.81it/s]


In [8]:
# %%
min_max_values_detailed = {}
for ens in min_max_values.keys():
    for var in min_max_values[ens].keys():
        try:
            min_max_values_detailed[var] = np.concatenate([min_max_values_detailed[var], min_max_values[ens][var]])
        except:
            min_max_values_detailed[var] = min_max_values[ens][var]


In [9]:
# %%
for var in min_max_values_detailed.keys():
    min_value = np.nanmin(min_max_values_detailed[var])
    max_value = np.nanmax(min_max_values_detailed[var])
    min_max_values_detailed[var] = {
        "minimum": min_value,
        "maximum": max_value
    }


In [10]:
# %%
df = pd.DataFrame(min_max_values_detailed)


In [11]:
# %%
ds_min_max = xr.Dataset.from_dataframe(df)
ds_min_max_dir = write_dir / "min-max-values"
ds_min_max_dir.mkdir(exist_ok=True, parents=True)
ds_min_max.to_netcdf(ds_min_max_dir / "min-max-values.nc")